# Analytic Eddy-Current Visualization

The original Mathematica current expressions are parsed into SymPy and evaluated on a Cartesian grid. Points exactly on analytic sector boundaries are excluded, and remaining non-finite boundary limits are set to zero before plotting.

# Corrientes analíticas


In [ ]:

from __future__ import annotations

import re
import warnings
from pathlib import Path
from functools import lru_cache

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import lambdify, Tuple
from sympy.parsing.mathematica import parse_mathematica
from sympy.utilities.exceptions import SymPyDeprecationWarning
from IPython.display import display, Image as IPyImage

warnings.filterwarnings("ignore", category=SymPyDeprecationWarning)


In [ ]:

# ============================================================
# PARÁMETROS QUE PUEDES CAMBIAR
# ============================================================
PARAMS = {
    'sigma': 3.1*10**7,
    'Omega': 2*np.pi*60,
    'omega': 0.003,
    'varphi': 0.000004,
    'B01': 0.0014,
    'B02': 0.00,
    'R11': 0.01,
    'R12': 0.02,
    'R21': 0.01,
    'R22': 0.02,
    'theta11': 0.0,
    'theta12': 3* np.pi /8,
    'theta21': 3* np.pi /8,
    'theta22': 3* np.pi /4,
    'R': 0.05,
}

# Mallado y visualización
N_GRID = 75
N_SNAPSHOTS = 6
RHO_EPS = 1e-7
QUIVER_STRIDE = 3
FIGSIZE = (14, 8)
CMAP = 'viridis'
QUIVER_COLOR = 'crimson'

# GIF
FPS = 10
N_ANIMATION_FRAMES = 30

# Carpeta de salida
OUTPUT_DIR = Path.cwd() / 'salidas_corrientes'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG = OUTPUT_DIR / 'corrientes_un_periodo.png'
OUTPUT_GIF = OUTPUT_DIR / 'corrientes_un_periodo.gif'


In [ ]:
MATHEMATICA_TEXT = r'''
ClearAll["Global`*"];
J11tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi])) B01 \[Omega] Cos[
      t \[CapitalOmega]] ((R11 - R12) (Sin[\[Theta] - \[Theta]11] -
          Sin[\[Theta] - \[Theta]12]) - \[Rho] Log[R12/
         R11] (Sin[2 (\[Theta] - \[Theta]11)] -
          Sin[2 (\[Theta] - \[Theta]12)]) -
       R11^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R11] Cos[
            2 (\[Theta] - \[Theta]11)])/
          R11^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R11] Cos[
            2 (\[Theta] - \[Theta]12)])/R11^2 +
          Sin[\[Theta] - \[Theta]11]/(
          2 R11) + (\[Rho] Sin[2 (\[Theta] - \[Theta]11)])/(
          2 R11^2) - (\[Rho]^2 ((2 \[Rho])/R11^2 - (
             2 Cos[\[Theta] - \[Theta]11])/R11) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 (1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R11)) - (\[Rho] Log[
            1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R11] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2) -
          Sin[\[Theta] - \[Theta]12]/(
          2 R11) - (\[Rho] Sin[2 (\[Theta] - \[Theta]12)])/(
          2 R11^2) + (\[Rho]^2 ((2 \[Rho])/R11^2 - (
             2 Cos[\[Theta] - \[Theta]12])/R11) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 (1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R11)) + (\[Rho] Log[
            1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R11] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R11^2) (-(Cos[\[Theta] - \[Theta]11]/R11) + (
             I Sin[\[Theta] - \[Theta]11])/R11) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R11] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R11^2) (-(Cos[\[Theta] - \[Theta]12]/R11) + (
             I Sin[\[Theta] - \[Theta]12])/R11) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R11]) -
       R11^2 ((
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^3) - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(2 R11 \[Rho]^2) - (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(2 R11 \[Rho]^2) + (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Cos[
            2 (\[Theta] - \[Theta]11)])/
          R12^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Cos[
            2 (\[Theta] - \[Theta]12)])/R12^2 +
          Sin[\[Theta] - \[Theta]11]/(
          2 R12) + (\[Rho] Sin[2 (\[Theta] - \[Theta]11)])/(
          2 R12^2) - (\[Rho]^2 ((2 \[Rho])/R12^2 - (
             2 Cos[\[Theta] - \[Theta]11])/R12) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R12^2 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12)) - (\[Rho] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2) -
          Sin[\[Theta] - \[Theta]12]/(
          2 R12) - (\[Rho] Sin[2 (\[Theta] - \[Theta]12)])/(
          2 R12^2) + (\[Rho]^2 ((2 \[Rho])/R12^2 - (
             2 Cos[\[Theta] - \[Theta]12])/R12) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12)) + (\[Rho] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R12^2) (-(Cos[\[Theta] - \[Theta]11]/R12) + (
             I Sin[\[Theta] - \[Theta]11])/R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R12^2) (-(Cos[\[Theta] - \[Theta]12]/R12) + (
             I Sin[\[Theta] - \[Theta]12])/R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
       R12^2 ((R^4 Arg[
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^3) - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(2 R12 \[Rho]^2) - (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(

          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (
          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(2 R12 \[Rho]^2) + (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(

             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), -(1/(
      2 \[Pi] \[Rho])) B01 \[Omega] Cos[
      t \[CapitalOmega]] ((R11 -
          R12) \[Rho] (Cos[\[Theta] - \[Theta]11] -
          Cos[\[Theta] - \[Theta]12]) -
       1/2 \[Rho]^2 (2 Cos[2 (\[Theta] - \[Theta]11)] -
          2 Cos[2 (\[Theta] - \[Theta]12)]) Log[R12/R11] -
       R11^2 ((\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R11) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/(
          2 R11^2) - (\[Rho] Cos[\[Theta] - \[Theta]12])/(
          2 R11) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/(
          2 R11^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R11])/(
          2 R11^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R11])/(

          2 R11^2) - (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R11] Sin[
            2 (\[Theta] - \[Theta]11)])/
          R11^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11^3 (1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R11)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R11] Sin[
            2 (\[Theta] - \[Theta]12)])/
          R11^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11^3 (1 + \[Rho]^2/R11^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R11)) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R11^2) ((I \[Rho] Cos[\[Theta] - \[Theta]11])/
             R11 + (\[Rho] Sin[\[Theta] - \[Theta]11])/
             R11) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R11] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R11^2) ((I \[Rho] Cos[\[Theta] - \[Theta]12])/
             R11 + (\[Rho] Sin[\[Theta] - \[Theta]12])/
             R11) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R11 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R11]) -
       R11^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R11 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]12])/(2 R11 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
          2 R11^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
          2 R11^2 \[Rho]^2) + (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 ((\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R12) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/(
          2 R12^2) - (\[Rho] Cos[\[Theta] - \[Theta]12])/(
          2 R12) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/(
          2 R12^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12])/(
          2 R12^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12])/(
          2 R12^2) - (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Sin[
            2 (\[Theta] - \[Theta]11)])/
          R12^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12^3 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R12)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Sin[
            2 (\[Theta] - \[Theta]12)])/
          R12^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12^3 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12)) +

          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]11])/
             R12 + (\[Rho] Sin[\[Theta] - \[Theta]11])/
             R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]12])/
             R12 + (\[Rho] Sin[\[Theta] - \[Theta]12])/
             R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
       R12^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R12 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]12])/(2 R12 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
          2 R12^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
          2 R12^2 \[Rho]^2) + (

          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -

          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), 0};
J12tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi])) B01 \[Omega] Cos[
      t \[CapitalOmega]] ((-\[Theta]11 + \[Theta]12) \[Rho] + (-R12 +
          R11^3/(3 \[Rho]^2) + (8 \[Rho])/
          3) (Sin[\[Theta] - \[Theta]11] -
          Sin[\[Theta] - \[Theta]12]) + (R11^4/(4 \[Rho]^3) + (
          3 \[Rho])/
          4 - \[Rho] Log[R12/\[Rho]]) (Sin[
           2 (\[Theta] - \[Theta]11)] -
          Sin[2 (\[Theta] - \[Theta]12)]) -
       8 \[Rho] (1/3 Sin[\[Theta] - \[Theta]11] -
          1/2 Arg[1 - Cos[\[Theta] - \[Theta]11] +
             I Sin[\[Theta] - \[Theta]11]] Sin[\[Theta] - \
\[Theta]11]^2 + 3/32 Sin[2 (\[Theta] - \[Theta]11)] -
          1/3 Sin[\[Theta] - \[Theta]12] +
          1/2 Arg[1 - Cos[\[Theta] - \[Theta]12] +

             I Sin[\[Theta] - \[Theta]12]] Sin[\[Theta] - \
\[Theta]12]^2 - 3/32 Sin[2 (\[Theta] - \[Theta]12)]) -
       R11^2 (-((\[Rho] Arg[
             1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]11)])/
           R11^2) + (\[Rho] Arg[
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]12)])/R11^2 -
          Sin[\[Theta] - \[Theta]11]/(2 R11) + (
          R11 Sin[\[Theta] - \[Theta]11])/(3 \[Rho]^2) + (
          R11^2 Sin[2 (\[Theta] - \[Theta]11)])/(
          4 \[Rho]^3) - (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (\[Rho] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2) +
          Sin[\[Theta] - \[Theta]12]/(2 R11) - (
          R11 Sin[\[Theta] - \[Theta]12])/(3 \[Rho]^2) - (
          R11^2 Sin[2 (\[Theta] - \[Theta]12)])/(
          4 \[Rho]^3) + (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (\[Rho] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R11^2) ((
             R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2 - (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R11^2) ((
             R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2 - (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]]) -
       R11^2 ((R^4 Arg[
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^3) - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(2 R11 \[Rho]^2) - (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(2 R11 \[Rho]^2) + (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Cos[
            2 (\[Theta] - \[Theta]11)])/
          R12^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Cos[
            2 (\[Theta] - \[Theta]12)])/R12^2 +
          Sin[\[Theta] - \[Theta]11]/(
          2 R12) + (\[Rho] Sin[2 (\[Theta] - \[Theta]11)])/(
          2 R12^2) - (\[Rho]^2 ((2 \[Rho])/R12^2 - (
             2 Cos[\[Theta] - \[Theta]11])/R12) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R12^2 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12)) - (\[Rho] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2) -
          Sin[\[Theta] - \[Theta]12]/(
          2 R12) - (\[Rho] Sin[2 (\[Theta] - \[Theta]12)])/(
          2 R12^2) + (\[Rho]^2 ((2 \[Rho])/R12^2 - (
             2 Cos[\[Theta] - \[Theta]12])/R12) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12)) + (\[Rho] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R12^2) (-(Cos[\[Theta] - \[Theta]11]/R12) + (
             I Sin[\[Theta] - \[Theta]11])/R12) Derivative[1][Arg][

            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R12^2) (-(Cos[\[Theta] - \[Theta]12]/R12) + (
             I Sin[\[Theta] - \[Theta]12])/R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
       R12^2 ((R^4 Arg[
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^3) - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(2 R12 \[Rho]^2) - (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (

          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(2 R12 \[Rho]^2) + (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), -(1/(
      2 \[Pi] \[Rho])) B01 \[Omega] Cos[
      t \[CapitalOmega]] ((-(R11^3/(3 \[Rho])) - R12 \[Rho] + (
          4 \[Rho]^2)/3) (Cos[\[Theta] - \[Theta]11] -
          Cos[\[Theta] - \[Theta]12]) + (2 Cos[
            2 (\[Theta] - \[Theta]11)] -
          2 Cos[2 (\[Theta] - \[Theta]12)]) (-(R11^4/(
           8 \[Rho]^2)) + \[Rho]^2/8 -
          1/2 \[Rho]^2 Log[R12/\[Rho]]) -
       4 \[Rho]^2 (1/3 Cos[\[Theta] - \[Theta]11] +
          3/16 Cos[2 (\[Theta] - \[Theta]11)] -
          1/3 Cos[\[Theta] - \[Theta]12] -
          3/16 Cos[2 (\[Theta] - \[Theta]12)] -
          Arg[1 - Cos[\[Theta] - \[Theta]11] +
             I Sin[\[Theta] - \[Theta]11]] Cos[\[Theta] - \[Theta]11] \
Sin[\[Theta] - \[Theta]11] +
          Arg[1 - Cos[\[Theta] - \[Theta]12] +
             I Sin[\[Theta] - \[Theta]12]] Cos[\[Theta] - \[Theta]12] \
Sin[\[Theta] - \[Theta]12] -
          1/2 Sin[\[Theta] - \[Theta]11]^2 (I Cos[\[Theta] - \
\[Theta]11] + Sin[\[Theta] - \[Theta]11]) Derivative[1][Arg][
            1 - Cos[\[Theta] - \[Theta]11] +
             I Sin[\[Theta] - \[Theta]11]] +
          1/2 Sin[\[Theta] - \[Theta]12]^2 (I Cos[\[Theta] - \
\[Theta]12] + Sin[\[Theta] - \[Theta]12]) Derivative[1][Arg][
            1 - Cos[\[Theta] - \[Theta]12] +
             I Sin[\[Theta] - \[Theta]12]]) -
       R11^2 (-((R11 Cos[\[Theta] - \[Theta]11])/(
           3 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R11) - (R11^2 Cos[2 (\[Theta] - \[Theta]11)])/(
          4 \[Rho]^2) + (R11 Cos[\[Theta] - \[Theta]12])/(
          3 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]12])/(2 R11) + (
          R11^2 Cos[2 (\[Theta] - \[Theta]12)])/(
          4 \[Rho]^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]])/(
          2 R11^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]])/(
          2 R11^2) + (\[Rho]^2 Arg[
            1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/
          R11^2 - (\[Rho] Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (\[Rho]^2 \
Arg[1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/
          R11^2 + (\[Rho] Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R11^2) ((
             I R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             R11 Sin[\[Theta] - \[Theta]11])/\[Rho]) Derivative[1][
            Arg][1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R11^2) ((
             I R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             R11 Sin[\[Theta] - \[Theta]12])/\[Rho]) Derivative[1][
            Arg][
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]]) -
       R11^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R11 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]12])/(2 R11 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
          2 R11^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
          2 R11^2 \[Rho]^2) + (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 ((\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R12) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/(
          2 R12^2) - (\[Rho] Cos[\[Theta] - \[Theta]12])/(
          2 R12) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/(
          2 R12^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12])/(
          2 R12^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12])/(
          2 R12^2) - (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Sin[
            2 (\[Theta] - \[Theta]11)])/
          R12^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12^3 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R12)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Sin[
            2 (\[Theta] - \[Theta]12)])/
          R12^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12^3 (1 + \[Rho]^2/R12^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12)) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
             R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]11])/
             R12 + (\[Rho] Sin[\[Theta] - \[Theta]11])/
             R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
             R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]12])/
             R12 + (\[Rho] Sin[\[Theta] - \[Theta]12])/
             R12) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
             I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
       R12^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R12 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]12])/(2 R12 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
          2 R12^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
          2 R12^2 \[Rho]^2) + (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), 0};
J13tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi])) B01 \[Omega] Cos[
      t \[CapitalOmega]] (-R11^2 (-((\[Rho] Arg[
             1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]11)])/R11^2) + (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(
          R11^2 \[Rho]^3) + (\[Rho] Arg[
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]12)])/R11^2 - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^3) -
          Sin[\[Theta] - \[Theta]11]/(2 R11) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(
          2 R11 \[Rho]^2) - (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R^2)) - (\[Rho] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2) + (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R11^2 \[Rho]^3) +
          Sin[\[Theta] - \[Theta]12]/(2 R11) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(
          2 R11 \[Rho]^2) + (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (
          R^4 ((2 R11^2 \[Rho])/R^4 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/
             R^2)) + (\[Rho] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2) - (
          R^4 Log[1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R11^2 \[Rho]^3) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R11^2) ((
             R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2 - (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R11^2) ((
             R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2 - (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R11 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 (-((\[Rho] Arg[
             1 - (R12 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R12 Sin[\[Theta] - \[Theta]11])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]11)])/R12^2) + (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
            2 (\[Theta] - \[Theta]11)])/(
          R12^2 \[Rho]^3) + (\[Rho] Arg[
            1 - (R12 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]12])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]12)])/R12^2 - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
            2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^3) -
          Sin[\[Theta] - \[Theta]11]/(2 R12) + (
          R^2 Sin[\[Theta] - \[Theta]11])/(
          2 R12 \[Rho]^2) - (\[Rho]^2 (-((2 R12^2)/\[Rho]^3) + (
             2 R12 Cos[\[Theta] - \[Theta]11])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R12^2 (1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
            2 (\[Theta] - \[Theta]11)])/(
          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R^2)) - (\[Rho] Log[
            1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2) + (
          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(2 R12^2 \[Rho]^3) +
          Sin[\[Theta] - \[Theta]12]/(2 R12) - (
          R^2 Sin[\[Theta] - \[Theta]12])/(
          2 R12 \[Rho]^2) + (\[Rho]^2 (-((2 R12^2)/\[Rho]^3) + (
             2 R12 Cos[\[Theta] - \[Theta]12])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 (1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (
          R^4 ((2 R12^2 \[Rho])/R^4 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
            2 (\[Theta] - \[Theta]12)])/(
          4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/
             R^2)) + (\[Rho] Log[
            1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2) - (
          R^4 Log[1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(2 R12^2 \[Rho]^3) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R12^2) ((
             R12 Cos[\[Theta] - \[Theta]11])/\[Rho]^2 - (
             I R12 Sin[\[Theta] - \[Theta]11])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R12 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]11])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]11])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]11])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R12^2) ((
             R12 Cos[\[Theta] - \[Theta]12])/\[Rho]^2 - (
             I R12 Sin[\[Theta] - \[Theta]12])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R12 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]12])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]12])/
              R^2) + (I R12 Sin[\[Theta] - \[Theta]12])/
             R^2) Derivative[1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), -(1/(
      2 \[Pi] \[Rho])) B01 \[Omega] Cos[
      t \[CapitalOmega]] (-R11^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(
           2 R11 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R11) + (R^2 Cos[\[Theta] - \[Theta]12])/(
          2 R11 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]12])/(
          2 R11) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]])/(2 R11^2) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
          2 R11^2 \[Rho]^2) + (\[Rho]^2 Cos[
            2 (\[Theta] - \[Theta]12)] Log[
            1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]])/(2 R11^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(

          2 R11^2 \[Rho]^2) + (\[Rho]^2 Arg[
            1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/R11^2 + (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          R11^2 \[Rho]^2) - (\[Rho] Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R^2)) - (\[Rho]^2 Arg[
            1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/R11^2 - (
          R^4 Arg[1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(

          R11^2 \[Rho]^2) + (\[Rho] Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11 (1 + R11^2/\[Rho]^2 - (
             2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
             2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R11^2) ((
             I R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             R11 Sin[\[Theta] - \[Theta]11])/\[Rho]) Derivative[1][
            Arg][1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R11^2) ((
             I R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             R11 Sin[\[Theta] - \[Theta]12])/\[Rho]) Derivative[1][
            Arg][1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R11^2 \[Rho]^2)) ((
             I R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
       R12^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(
           2 R12 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]11])/(
          2 R12) + (R^2 Cos[\[Theta] - \[Theta]12])/(
          2 R12 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]12])/(
          2 R12) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/\[Rho]])/(2 R12^2) - (
          R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(

          2 R12^2 \[Rho]^2) + (\[Rho]^2 Cos[
            2 (\[Theta] - \[Theta]12)] Log[
            1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/\[Rho]])/(2 R12^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
            1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
          2 R12^2 \[Rho]^2) + (\[Rho]^2 Arg[
            1 - (R12 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]11])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]11)])/R12^2 + (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          R12^2 \[Rho]^2) - (\[Rho] Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12 (1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (
          R^2 Sin[\[Theta] - \[Theta]11] Sin[
            2 (\[Theta] - \[Theta]11)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/
             R^2)) - (\[Rho]^2 Arg[
            1 - (R12 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]12])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]12)])/R12^2 - (
          R^4 Arg[1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          R12^2 \[Rho]^2) + (\[Rho] Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12 (1 + R12^2/\[Rho]^2 - (
             2 R12 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (
          R^2 Sin[\[Theta] - \[Theta]12] Sin[
            2 (\[Theta] - \[Theta]12)])/(
          2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
             2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/R12^2) ((
             I R12 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             R12 Sin[\[Theta] - \[Theta]11])/\[Rho]) Derivative[1][
            Arg][1 - (R12 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]11])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/R12^2) ((
             I R12 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             R12 Sin[\[Theta] - \[Theta]12])/\[Rho]) Derivative[1][
            Arg][1 - (R12 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
             I R12 Sin[\[Theta] - \[Theta]12])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
             R12^2 \[Rho]^2)) ((
             I R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
            1][Arg][
            1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
             I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), 0};

J14tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi])) B01 \[Omega] Cos[
       t \[CapitalOmega]] ((-\[Theta]11 + \[Theta]12) \[Rho] + (-R12 +
            R11^3/(3 \[Rho]^2) + (8 \[Rho])/
           3) (Sin[\[Theta] - \[Theta]11] -
           Sin[\[Theta] - \[Theta]12]) + (R11^4/(4 \[Rho]^3) + (
           3 \[Rho])/
           4 - \[Rho] Log[R12/\[Rho]]) (Sin[
            2 (\[Theta] - \[Theta]11)] -
           Sin[2 (\[Theta] - \[Theta]12)]) -
        8 \[Rho] (1/3 Sin[\[Theta] - \[Theta]11] -
           1/2 Arg[
             1 - Cos[\[Theta] - \[Theta]11] +
              I Sin[\[Theta] - \[Theta]11]] Sin[\[Theta] - \
\[Theta]11]^2 + 3/32 Sin[2 (\[Theta] - \[Theta]11)] -
           1/3 Sin[\[Theta] - \[Theta]12] +
           1/2 Arg[
             1 - Cos[\[Theta] - \[Theta]12] +
              I Sin[\[Theta] - \[Theta]12]] Sin[\[Theta] - \
\[Theta]12]^2 - 3/32 Sin[2 (\[Theta] - \[Theta]12)]) -
        R11^2 (-((\[Rho] Arg[
              1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
               I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Cos[
              2 (\[Theta] - \[Theta]11)])/
            R11^2) + (\[Rho] Arg[
             1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]12)])/R11^2 -
           Sin[\[Theta] - \[Theta]11]/(2 R11) + (
           R11 Sin[\[Theta] - \[Theta]11])/(3 \[Rho]^2) + (
           R11^2 Sin[2 (\[Theta] - \[Theta]11)])/(

           4 \[Rho]^3) - (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
              2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2) Sin[
             2 (\[Theta] - \[Theta]11)])/(
           4 R11^2 (1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (\[Rho] \
Log[1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]11)])/(2 R11^2) +
           Sin[\[Theta] - \[Theta]12]/(2 R11) - (
           R11 Sin[\[Theta] - \[Theta]12])/(3 \[Rho]^2) - (
           R11^2 Sin[2 (\[Theta] - \[Theta]12)])/(
           4 \[Rho]^3) + (\[Rho]^2 (-((2 R11^2)/\[Rho]^3) + (
              2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2) Sin[
             2 (\[Theta] - \[Theta]12)])/(
           4 R11^2 (1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) + (\[Rho] \
Log[1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]12)])/(2 R11^2) +
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
              R11^2) ((R11 Cos[\[Theta] - \[Theta]11])/\[Rho]^2 - (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]^2) Derivative[
             1][Arg][
             1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] -
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
              R11^2) ((R11 Cos[\[Theta] - \[Theta]12])/\[Rho]^2 - (
              I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]^2) Derivative[
             1][Arg][
             1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]]) -
        R11^2 ((R^4 Arg[
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
             2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^3) - (
           R^4 Arg[
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
             2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^3) + (
           R^2 Sin[\[Theta] - \[Theta]11])/(2 R11 \[Rho]^2) - (
           R^4 ((2 R11^2 \[Rho])/R^4 - (
              2 R11 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
             2 (\[Theta] - \[Theta]11)])/(
           4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (

           R^4 Log[
             1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
             2 (\[Theta] - \[Theta]11)])/(2 R11^2 \[Rho]^3) - (
           R^2 Sin[\[Theta] - \[Theta]12])/(2 R11 \[Rho]^2) + (
           R^4 ((2 R11^2 \[Rho])/R^4 - (
              2 R11 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
             2 (\[Theta] - \[Theta]12)])/(
           4 R11^2 \[Rho]^2 (1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
           R^4 Log[
             1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
             2 (\[Theta] - \[Theta]12)])/(2 R11^2 \[Rho]^3) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
              R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]11])/
               R^2) + (I R11 Sin[\[Theta] - \[Theta]11])/
              R^2) Derivative[1][Arg][
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
              R11^2 \[Rho]^2)) (-((R11 Cos[\[Theta] - \[Theta]12])/
               R^2) + (I R11 Sin[\[Theta] - \[Theta]12])/
              R^2) Derivative[1][Arg][
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
        R12^2 ((\[Rho] Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Cos[
             2 (\[Theta] - \[Theta]11)])/
           R12^2 - (\[Rho] Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Cos[
             2 (\[Theta] - \[Theta]12)])/R12^2 +
           Sin[\[Theta] - \[Theta]11]/(
           2 R12) + (\[Rho] Sin[2 (\[Theta] - \[Theta]11)])/(
           2 R12^2) - (\[Rho]^2 ((2 \[Rho])/R12^2 - (
              2 Cos[\[Theta] - \[Theta]11])/R12) Sin[
             2 (\[Theta] - \[Theta]11)])/(
           4 R12^2 (1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]11])/
              R12)) - (\[Rho] Log[
             1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12] Sin[
             2 (\[Theta] - \[Theta]11)])/(2 R12^2) -
           Sin[\[Theta] - \[Theta]12]/(
           2 R12) - (\[Rho] Sin[2 (\[Theta] - \[Theta]12)])/(
           2 R12^2) + (\[Rho]^2 ((2 \[Rho])/R12^2 - (
              2 Cos[\[Theta] - \[Theta]12])/R12) Sin[
             2 (\[Theta] - \[Theta]12)])/(
           4 R12^2 (1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]12])/
              R12)) + (\[Rho] Log[
             1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12] Sin[
             2 (\[Theta] - \[Theta]12)])/(2 R12^2) +
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
              R12^2) (-(Cos[\[Theta] - \[Theta]11]/R12) + (
              I Sin[\[Theta] - \[Theta]11])/R12) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
              R12^2) (-(Cos[\[Theta] - \[Theta]12]/R12) + (
              I Sin[\[Theta] - \[Theta]12])/R12) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
        R12^2 ((R^4 Arg[
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Cos[
             2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^3) - (

           R^4 Arg[
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Cos[
             2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^3) + (
           R^2 Sin[\[Theta] - \[Theta]11])/(2 R12 \[Rho]^2) - (
           R^4 ((2 R12^2 \[Rho])/R^4 - (
              2 R12 Cos[\[Theta] - \[Theta]11])/R^2) Sin[
             2 (\[Theta] - \[Theta]11)])/(
           4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) + (
           R^4 Log[
             1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2] Sin[
             2 (\[Theta] - \[Theta]11)])/(2 R12^2 \[Rho]^3) - (
           R^2 Sin[\[Theta] - \[Theta]12])/(2 R12 \[Rho]^2) + (
           R^4 ((2 R12^2 \[Rho])/R^4 - (
              2 R12 Cos[\[Theta] - \[Theta]12])/R^2) Sin[
             2 (\[Theta] - \[Theta]12)])/(
           4 R12^2 \[Rho]^2 (1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) - (
           R^4 Log[
             1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2] Sin[
             2 (\[Theta] - \[Theta]12)])/(2 R12^2 \[Rho]^3) +

           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
              R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]11])/
               R^2) + (I R12 Sin[\[Theta] - \[Theta]11])/
              R^2) Derivative[1][Arg][
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
              R12^2 \[Rho]^2)) (-((R12 Cos[\[Theta] - \[Theta]12])/
               R^2) + (I R12 Sin[\[Theta] - \[Theta]12])/
              R^2) Derivative[1][Arg][
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])), -(1/(
       2 \[Pi] \[Rho])) B01 \[Omega] Cos[
       t \[CapitalOmega]] ((-(R11^3/(3 \[Rho])) - R12 \[Rho] + (
           4 \[Rho]^2)/3) (Cos[\[Theta] - \[Theta]11] -
           Cos[\[Theta] - \[Theta]12]) + (2 Cos[
             2 (\[Theta] - \[Theta]11)] -
           2 Cos[2 (\[Theta] - \[Theta]12)]) (-(R11^4/(
            8 \[Rho]^2)) + \[Rho]^2/8 -
           1/2 \[Rho]^2 Log[R12/\[Rho]]) -
        4 \[Rho]^2 (1/3 Cos[\[Theta] - \[Theta]11] +
           3/16 Cos[2 (\[Theta] - \[Theta]11)] -
           1/3 Cos[\[Theta] - \[Theta]12] -
           3/16 Cos[2 (\[Theta] - \[Theta]12)] -
           Arg[1 - Cos[\[Theta] - \[Theta]11] +
              I Sin[\[Theta] - \[Theta]11]] Cos[\[Theta] - \
\[Theta]11] Sin[\[Theta] - \[Theta]11] +
           Arg[1 - Cos[\[Theta] - \[Theta]12] +
              I Sin[\[Theta] - \[Theta]12]] Cos[\[Theta] - \
\[Theta]12] Sin[\[Theta] - \[Theta]12] -
           1/2 Sin[\[Theta] - \[Theta]11]^2 (I Cos[\[Theta] - \
\[Theta]11] + Sin[\[Theta] - \[Theta]11]) Derivative[1][Arg][
             1 - Cos[\[Theta] - \[Theta]11] +
              I Sin[\[Theta] - \[Theta]11]] +
           1/2 Sin[\[Theta] - \[Theta]12]^2 (I Cos[\[Theta] - \
\[Theta]12] + Sin[\[Theta] - \[Theta]12]) Derivative[1][Arg][
             1 - Cos[\[Theta] - \[Theta]12] +
              I Sin[\[Theta] - \[Theta]12]]) -
        R11^2 (-((R11 Cos[\[Theta] - \[Theta]11])/(
            3 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]11])/(
           2 R11) - (R11^2 Cos[2 (\[Theta] - \[Theta]11)])/(
           4 \[Rho]^2) + (R11 Cos[\[Theta] - \[Theta]12])/(
           3 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]12])/(2 R11) + (
           R11^2 Cos[2 (\[Theta] - \[Theta]12)])/(
           4 \[Rho]^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
             1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho]])/(
           2 R11^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
             1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho]])/(
           2 R11^2) + (\[Rho]^2 Arg[
             1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]11)])/
           R11^2 - (\[Rho] Sin[\[Theta] - \[Theta]11] Sin[
             2 (\[Theta] - \[Theta]11)])/(
           2 R11 (1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]11])/\[Rho])) - (\[Rho]^2 \
Arg[1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]12)])/
           R11^2 + (\[Rho] Sin[\[Theta] - \[Theta]12] Sin[
             2 (\[Theta] - \[Theta]12)])/(
           2 R11 (1 + R11^2/\[Rho]^2 - (
              2 R11 Cos[\[Theta] - \[Theta]12])/\[Rho])) +
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
              R11^2) ((I R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              R11 Sin[\[Theta] - \[Theta]11])/\[Rho]) Derivative[1][
             Arg][1 - (R11 Cos[\[Theta] - \[Theta]11])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]11])/\[Rho]] -
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
              R11^2) ((I R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
              R11 Sin[\[Theta] - \[Theta]12])/\[Rho]) Derivative[1][
             Arg][1 - (R11 Cos[\[Theta] - \[Theta]12])/\[Rho] + (
              I R11 Sin[\[Theta] - \[Theta]12])/\[Rho]]) -
        R11^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R11 \[Rho])) + (
           R^2 Cos[\[Theta] - \[Theta]12])/(2 R11 \[Rho]) - (
           R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
             1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
           2 R11^2 \[Rho]^2) + (
           R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
             1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
           2 R11^2 \[Rho]^2) + (
           R^4 Arg[
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
             2 (\[Theta] - \[Theta]11)])/(R11^2 \[Rho]^2) - (
           R^2 Sin[\[Theta] - \[Theta]11] Sin[
             2 (\[Theta] - \[Theta]11)])/(
           2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
           R^4 Arg[
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
             2 (\[Theta] - \[Theta]12)])/(R11^2 \[Rho]^2) + (
           R^2 Sin[\[Theta] - \[Theta]12] Sin[
             2 (\[Theta] - \[Theta]12)])/(
           2 R11 \[Rho] (1 + (R11^2 \[Rho]^2)/R^4 - (
              2 R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
              R11^2 \[Rho]^2)) ((
              I R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
             1][Arg][
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
              R11^2 \[Rho]^2)) ((
              I R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
             1][Arg][
             1 - (R11 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R11 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2]) +
        R12^2 ((\[Rho] Cos[\[Theta] - \[Theta]11])/(
           2 R12) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/(
           2 R12^2) - (\[Rho] Cos[\[Theta] - \[Theta]12])/(
           2 R12) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/(
           2 R12^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)] Log[
             1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]11])/R12])/(
           2 R12^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)] Log[
             1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12])/(
           2 R12^2) - (\[Rho]^2 Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] Sin[
             2 (\[Theta] - \[Theta]11)])/
           R12^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]11] Sin[
             2 (\[Theta] - \[Theta]11)])/(
           2 R12^3 (1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]11])/
              R12)) + (\[Rho]^2 Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]12])/R12] Sin[
             2 (\[Theta] - \[Theta]12)])/
           R12^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]12] Sin[
             2 (\[Theta] - \[Theta]12)])/(
           2 R12^3 (1 + \[Rho]^2/R12^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]12])/R12)) +
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]11)])/
              R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]11])/
              R12 + (\[Rho] Sin[\[Theta] - \[Theta]11])/
              R12) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]11])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]11])/R12] -
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]12)])/
              R12^2) ((I \[Rho] Cos[\[Theta] - \[Theta]12])/
              R12 + (\[Rho] Sin[\[Theta] - \[Theta]12])/
              R12) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]12])/R12 + (
              I \[Rho] Sin[\[Theta] - \[Theta]12])/R12]) +
        R12^2 (-((R^2 Cos[\[Theta] - \[Theta]11])/(2 R12 \[Rho])) + (
           R^2 Cos[\[Theta] - \[Theta]12])/(2 R12 \[Rho]) - (
           R^4 Cos[2 (\[Theta] - \[Theta]11)] Log[
             1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2])/(
           2 R12^2 \[Rho]^2) + (
           R^4 Cos[2 (\[Theta] - \[Theta]12)] Log[
             1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2])/(
           2 R12^2 \[Rho]^2) + (
           R^4 Arg[
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] Sin[
             2 (\[Theta] - \[Theta]11)])/(R12^2 \[Rho]^2) - (
           R^2 Sin[\[Theta] - \[Theta]11] Sin[
             2 (\[Theta] - \[Theta]11)])/(
           2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2)) - (
           R^4 Arg[
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2] Sin[
             2 (\[Theta] - \[Theta]12)])/(R12^2 \[Rho]^2) + (
           R^2 Sin[\[Theta] - \[Theta]12] Sin[
             2 (\[Theta] - \[Theta]12)])/(
           2 R12 \[Rho] (1 + (R12^2 \[Rho]^2)/R^4 - (
              2 R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2)) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]11)])/(
              R12^2 \[Rho]^2)) ((
              I R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2) Derivative[
             1][Arg][
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]11])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]11])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]12)])/(
              R12^2 \[Rho]^2)) ((
              I R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2) Derivative[
             1][Arg][
             1 - (R12 \[Rho] Cos[\[Theta] - \[Theta]12])/R^2 + (
              I R12 \[Rho] Sin[\[Theta] - \[Theta]12])/R^2])),
     0} + \[Sigma] Cross[{0, \[Rho] \[Omega], 0}, {0, 0,
      B01 Cos[\[CapitalOmega] t]}];

J21tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] ((R21 - R22) (Sin[\[Theta] - \[Theta]21] -
          Sin[\[Theta] - \[Theta]22]) - \[Rho] Log[R22/
         R21] (Sin[2 (\[Theta] - \[Theta]21)] -
          Sin[2 (\[Theta] - \[Theta]22)]) -
       R21^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R21] Cos[
            2 (\[Theta] - \[Theta]21)])/
          R21^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R21] Cos[
            2 (\[Theta] - \[Theta]22)])/R21^2 +
          Sin[\[Theta] - \[Theta]21]/(
          2 R21) + (\[Rho] Sin[2 (\[Theta] - \[Theta]21)])/(
          2 R21^2) - (\[Rho]^2 ((2 \[Rho])/R21^2 - (
             2 Cos[\[Theta] - \[Theta]21])/R21) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 (1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R21)) - (\[Rho] Log[
            1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R21] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2) -
          Sin[\[Theta] - \[Theta]22]/(
          2 R21) - (\[Rho] Sin[2 (\[Theta] - \[Theta]22)])/(
          2 R21^2) + (\[Rho]^2 ((2 \[Rho])/R21^2 - (
             2 Cos[\[Theta] - \[Theta]22])/R21) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 (1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R21)) + (\[Rho] Log[
            1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R21] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R21^2) (-(Cos[\[Theta] - \[Theta]21]/R21) + (
             I Sin[\[Theta] - \[Theta]21])/R21) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R21] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R21^2) (-(Cos[\[Theta] - \[Theta]22]/R21) + (
             I Sin[\[Theta] - \[Theta]22])/R21) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R21]) -
       R21^2 ((R^4 Arg[
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^3) - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(2 R21 \[Rho]^2) - (
          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(2 R21 \[Rho]^2) + (
          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Cos[
            2 (\[Theta] - \[Theta]21)])/
          R22^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Cos[
            2 (\[Theta] - \[Theta]22)])/R22^2 +
          Sin[\[Theta] - \[Theta]21]/(
          2 R22) + (\[Rho] Sin[2 (\[Theta] - \[Theta]21)])/(
          2 R22^2) - (\[Rho]^2 ((2 \[Rho])/R22^2 - (
             2 Cos[\[Theta] - \[Theta]21])/R22) Sin[
            2 (\[Theta] - \[Theta]21)])/(

          4 R22^2 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22)) - (\[Rho] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2) -
          Sin[\[Theta] - \[Theta]22]/(
          2 R22) - (\[Rho] Sin[2 (\[Theta] - \[Theta]22)])/(
          2 R22^2) + (\[Rho]^2 ((2 \[Rho])/R22^2 - (
             2 Cos[\[Theta] - \[Theta]22])/R22) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R22^2 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22)) + (\[Rho] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R22^2) (-(Cos[\[Theta] - \[Theta]21]/R22) + (
             I Sin[\[Theta] - \[Theta]21])/R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R22^2) (-(Cos[\[Theta] - \[Theta]22]/R22) + (
             I Sin[\[Theta] - \[Theta]22])/R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
       R22^2 ((R^4 Arg[
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^3) - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(2 R22 \[Rho]^2) - (
          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(2 R22 \[Rho]^2) + (

          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), -(1/(
      2 \[Pi] \[Rho]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] ((R21 -
          R22) \[Rho] (Cos[\[Theta] - \[Theta]21] -
          Cos[\[Theta] - \[Theta]22]) -
       1/2 \[Rho]^2 (2 Cos[2 (\[Theta] - \[Theta]21)] -
          2 Cos[2 (\[Theta] - \[Theta]22)]) Log[R22/R21] -
       R21^2 ((\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R21) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/(
          2 R21^2) - (\[Rho] Cos[\[Theta] - \[Theta]22])/(
          2 R21) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/(
          2 R21^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R21])/(
          2 R21^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R21])/(
          2 R21^2) - (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R21] Sin[
            2 (\[Theta] - \[Theta]21)])/
          R21^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21^3 (1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R21)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R21] Sin[
            2 (\[Theta] - \[Theta]22)])/
          R21^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21^3 (1 + \[Rho]^2/R21^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R21)) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R21^2) ((I \[Rho] Cos[\[Theta] - \[Theta]21])/
             R21 + (\[Rho] Sin[\[Theta] - \[Theta]21])/
             R21) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R21] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R21^2) ((I \[Rho] Cos[\[Theta] - \[Theta]22])/
             R21 + (\[Rho] Sin[\[Theta] - \[Theta]22])/
             R21) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R21 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R21]) -
       R21^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R21 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]22])/(2 R21 \[Rho]) - (

          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R21^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R21^2 \[Rho]^2) + (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +

          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 ((\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R22) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/(
          2 R22^2) - (\[Rho] Cos[\[Theta] - \[Theta]22])/(
          2 R22) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/(
          2 R22^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22])/(
          2 R22^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[

            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22])/(
          2 R22^2) - (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Sin[
            2 (\[Theta] - \[Theta]21)])/
          R22^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22^3 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R22)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Sin[
            2 (\[Theta] - \[Theta]22)])/
          R22^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22^3 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22)) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]21])/
             R22 + (\[Rho] Sin[\[Theta] - \[Theta]21])/
             R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -

          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]22])/
             R22 + (\[Rho] Sin[\[Theta] - \[Theta]22])/
             R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
       R22^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R22 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]22])/(2 R22 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R22^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R22^2 \[Rho]^2) + (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), 0};
J22tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] ((-\[Theta]21 + \[Theta]22) \[Rho] + (-R22 +
           R21^3/(3 \[Rho]^2) + (8 \[Rho])/
          3) (Sin[\[Theta] - \[Theta]21] -
          Sin[\[Theta] - \[Theta]22]) + (R21^4/(4 \[Rho]^3) + (
          3 \[Rho])/
          4 - \[Rho] Log[R22/\[Rho]]) (Sin[
           2 (\[Theta] - \[Theta]21)] -
          Sin[2 (\[Theta] - \[Theta]22)]) -
       8 \[Rho] (1/3 Sin[\[Theta] - \[Theta]21] -
          1/2 Arg[1 - Cos[\[Theta] - \[Theta]21] +
             I Sin[\[Theta] - \[Theta]21]] Sin[\[Theta] - \
\[Theta]21]^2 + 3/32 Sin[2 (\[Theta] - \[Theta]21)] -
          1/3 Sin[\[Theta] - \[Theta]22] +
          1/2 Arg[1 - Cos[\[Theta] - \[Theta]22] +
             I Sin[\[Theta] - \[Theta]22]] Sin[\[Theta] - \
\[Theta]22]^2 - 3/32 Sin[2 (\[Theta] - \[Theta]22)]) -
       R21^2 (-((\[Rho] Arg[
             1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]21)])/
           R21^2) + (\[Rho] Arg[
            1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]22)])/R21^2 -
          Sin[\[Theta] - \[Theta]21]/(2 R21) + (
          R21 Sin[\[Theta] - \[Theta]21])/(3 \[Rho]^2) + (
          R21^2 Sin[2 (\[Theta] - \[Theta]21)])/(
          4 \[Rho]^3) - (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (\[Rho] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2) +
          Sin[\[Theta] - \[Theta]22]/(2 R21) - (
          R21 Sin[\[Theta] - \[Theta]22])/(3 \[Rho]^2) - (
          R21^2 Sin[2 (\[Theta] - \[Theta]22)])/(
          4 \[Rho]^3) + (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (\[Rho] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R21^2) ((
             R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2 - (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R21^2) ((
             R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2 - (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]]) -
       R21^2 ((R^4 Arg[
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^3) - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(2 R21 \[Rho]^2) - (

          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(2 R21 \[Rho]^2) + (
          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 ((\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Cos[
            2 (\[Theta] - \[Theta]21)])/
          R22^2 - (\[Rho] Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Cos[
            2 (\[Theta] - \[Theta]22)])/R22^2 +
          Sin[\[Theta] - \[Theta]21]/(
          2 R22) + (\[Rho] Sin[2 (\[Theta] - \[Theta]21)])/(
          2 R22^2) - (\[Rho]^2 ((2 \[Rho])/R22^2 - (
             2 Cos[\[Theta] - \[Theta]21])/R22) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R22^2 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22)) - (\[Rho] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2) -
          Sin[\[Theta] - \[Theta]22]/(
          2 R22) - (\[Rho] Sin[2 (\[Theta] - \[Theta]22)])/(
          2 R22^2) + (\[Rho]^2 ((2 \[Rho])/R22^2 - (
             2 Cos[\[Theta] - \[Theta]22])/R22) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R22^2 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22)) + (\[Rho] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R22^2) (-(Cos[\[Theta] - \[Theta]21]/R22) + (
             I Sin[\[Theta] - \[Theta]21])/R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R22^2) (-(Cos[\[Theta] - \[Theta]22]/R22) + (
             I Sin[\[Theta] - \[Theta]22])/R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
       R22^2 ((R^4 Arg[
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^3) - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^3) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(2 R22 \[Rho]^2) - (
          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2 \[Rho]^3) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(2 R22 \[Rho]^2) + (
          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(

          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2 \[Rho]^3) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), -(1/(
      2 \[Pi] \[Rho]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] ((-(R21^3/(3 \[Rho])) - R22 \[Rho] + (
          4 \[Rho]^2)/3) (Cos[\[Theta] - \[Theta]21] -

          Cos[\[Theta] - \[Theta]22]) + (2 Cos[
            2 (\[Theta] - \[Theta]21)] -
          2 Cos[2 (\[Theta] - \[Theta]22)]) (-(R21^4/(
           8 \[Rho]^2)) + \[Rho]^2/8 -
          1/2 \[Rho]^2 Log[R22/\[Rho]]) -
       4 \[Rho]^2 (1/3 Cos[\[Theta] - \[Theta]21] +
          3/16 Cos[2 (\[Theta] - \[Theta]21)] -
          1/3 Cos[\[Theta] - \[Theta]22] -
          3/16 Cos[2 (\[Theta] - \[Theta]22)] -
          Arg[1 - Cos[\[Theta] - \[Theta]21] +
             I Sin[\[Theta] - \[Theta]21]] Cos[\[Theta] - \[Theta]21] \
Sin[\[Theta] - \[Theta]21] +
          Arg[1 - Cos[\[Theta] - \[Theta]22] +
             I Sin[\[Theta] - \[Theta]22]] Cos[\[Theta] - \[Theta]22] \
Sin[\[Theta] - \[Theta]22] -
          1/2 Sin[\[Theta] - \[Theta]21]^2 (I Cos[\[Theta] - \
\[Theta]21] + Sin[\[Theta] - \[Theta]21]) Derivative[1][Arg][
            1 - Cos[\[Theta] - \[Theta]21] +
             I Sin[\[Theta] - \[Theta]21]] +
          1/2 Sin[\[Theta] - \[Theta]22]^2 (I Cos[\[Theta] - \
\[Theta]22] + Sin[\[Theta] - \[Theta]22]) Derivative[1][Arg][
            1 - Cos[\[Theta] - \[Theta]22] +
             I Sin[\[Theta] - \[Theta]22]]) -
       R21^2 (-((R21 Cos[\[Theta] - \[Theta]21])/(
           3 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R21) - (R21^2 Cos[2 (\[Theta] - \[Theta]21)])/(
          4 \[Rho]^2) + (R21 Cos[\[Theta] - \[Theta]22])/(
          3 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]22])/(2 R21) + (
          R21^2 Cos[2 (\[Theta] - \[Theta]22)])/(
          4 \[Rho]^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]])/(
          2 R21^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]])/(
          2 R21^2) + (\[Rho]^2 Arg[
            1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/
          R21^2 - (\[Rho] Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (\[Rho]^2 \
Arg[1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/
          R21^2 + (\[Rho] Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R21^2) ((
             I R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             R21 Sin[\[Theta] - \[Theta]21])/\[Rho]) Derivative[1][
            Arg][1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R21^2) ((
             I R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             R21 Sin[\[Theta] - \[Theta]22])/\[Rho]) Derivative[1][
            Arg][1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]]) -
       R21^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R21 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]22])/(2 R21 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R21^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R21^2 \[Rho]^2) + (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 ((\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R22) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/(
          2 R22^2) - (\[Rho] Cos[\[Theta] - \[Theta]22])/(
          2 R22) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/(
          2 R22^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22])/(
          2 R22^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22])/(
          2 R22^2) - (\[Rho]^2 Arg[

            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Sin[
            2 (\[Theta] - \[Theta]21)])/
          R22^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22^3 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R22)) + (\[Rho]^2 Arg[
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Sin[
            2 (\[Theta] - \[Theta]22)])/
          R22^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22^3 (1 + \[Rho]^2/R22^2 - (
             2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22)) +
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
             R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]21])/
             R22 + (\[Rho] Sin[\[Theta] - \[Theta]21])/
             R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -
          1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
             R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]22])/
             R22 + (\[Rho] Sin[\[Theta] - \[Theta]22])/
             R22) Derivative[1][Arg][
            1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
             I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
       R22^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R22 \[Rho])) + (
          R^2 Cos[\[Theta] - \[Theta]22])/(2 R22 \[Rho]) - (
          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R22^2 \[Rho]^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R22^2 \[Rho]^2) + (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^2) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^2) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), 0};
J23tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] (-R21^2 (-((\[Rho] Arg[
             1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]21)])/R21^2) + (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(
          R21^2 \[Rho]^3) + (\[Rho] Arg[
            1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]22)])/R21^2 - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^3) -
          Sin[\[Theta] - \[Theta]21]/(2 R21) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(
          2 R21 \[Rho]^2) - (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (

          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R^2)) - (\[Rho] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2) + (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R21^2 \[Rho]^3) +
          Sin[\[Theta] - \[Theta]22]/(2 R21) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(
          2 R21 \[Rho]^2) + (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (
          R^4 ((2 R21^2 \[Rho])/R^4 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/
             R^2)) + (\[Rho] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2) - (
          R^4 Log[1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R21^2 \[Rho]^3) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R21^2) ((
             R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2 - (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R21^2) ((
             R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2 - (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R21 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 (-((\[Rho] Arg[
             1 - (R22 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R22 Sin[\[Theta] - \[Theta]21])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]21)])/R22^2) + (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
            2 (\[Theta] - \[Theta]21)])/(
          R22^2 \[Rho]^3) + (\[Rho] Arg[
            1 - (R22 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]22])/\[Rho]] Cos[
            2 (\[Theta] - \[Theta]22)])/R22^2 - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
            2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^3) -
          Sin[\[Theta] - \[Theta]21]/(2 R22) + (
          R^2 Sin[\[Theta] - \[Theta]21])/(
          2 R22 \[Rho]^2) - (\[Rho]^2 (-((2 R22^2)/\[Rho]^3) + (
             2 R22 Cos[\[Theta] - \[Theta]21])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R22^2 (1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (
          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
            2 (\[Theta] - \[Theta]21)])/(
          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R^2)) - (\[Rho] Log[
            1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2) + (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(2 R22^2 \[Rho]^3) +
          Sin[\[Theta] - \[Theta]22]/(2 R22) - (
          R^2 Sin[\[Theta] - \[Theta]22])/(
          2 R22 \[Rho]^2) + (\[Rho]^2 (-((2 R22^2)/\[Rho]^3) + (
             2 R22 Cos[\[Theta] - \[Theta]22])/\[Rho]^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R22^2 (1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (
          R^4 ((2 R22^2 \[Rho])/R^4 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
            2 (\[Theta] - \[Theta]22)])/(
          4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/
             R^2)) + (\[Rho] Log[
            1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2) - (
          R^4 Log[1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(2 R22^2 \[Rho]^3) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R22^2) ((
             R22 Cos[\[Theta] - \[Theta]21])/\[Rho]^2 - (
             I R22 Sin[\[Theta] - \[Theta]21])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R22 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]21])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(

             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]21])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]21])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R22^2) ((
             R22 Cos[\[Theta] - \[Theta]22])/\[Rho]^2 - (
             I R22 Sin[\[Theta] - \[Theta]22])/\[Rho]^2) Derivative[
            1][Arg][
            1 - (R22 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]22])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]22])/
              R^2) + (I R22 Sin[\[Theta] - \[Theta]22])/
             R^2) Derivative[1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), -(1/(
      2 \[Pi] \[Rho]))
       B02 \[Omega] Cos[\[CurlyPhi] +
       t \[CapitalOmega]] (-R21^2 (-((
           R^2 Cos[\[Theta] - \[Theta]21])/(
           2 R21 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R21) + (R^2 Cos[\[Theta] - \[Theta]22])/(
          2 R21 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]22])/(
          2 R21) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]])/(2 R21^2) - (
          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R21^2 \[Rho]^2) + (\[Rho]^2 Cos[
            2 (\[Theta] - \[Theta]22)] Log[
            1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]])/(2 R21^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R21^2 \[Rho]^2) + (\[Rho]^2 Arg[
            1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/R21^2 + (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(

          R21^2 \[Rho]^2) - (\[Rho] Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R^2)) - (\[Rho]^2 Arg[
            1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/R21^2 - (
          R^4 Arg[1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          R21^2 \[Rho]^2) + (\[Rho] Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21 (1 + R21^2/\[Rho]^2 - (
             2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
             2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +

          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R21^2) ((
             I R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             R21 Sin[\[Theta] - \[Theta]21])/\[Rho]) Derivative[1][
            Arg][1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R21^2) ((
             I R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             R21 Sin[\[Theta] - \[Theta]22])/\[Rho]) Derivative[1][
            Arg][1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R21^2 \[Rho]^2)) ((
             I R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
       R22^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(
           2 R22 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]21])/(
          2 R22) + (R^2 Cos[\[Theta] - \[Theta]22])/(
          2 R22 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]22])/(
          2 R22) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/\[Rho]])/(2 R22^2) - (
          R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
          2 R22^2 \[Rho]^2) + (\[Rho]^2 Cos[
            2 (\[Theta] - \[Theta]22)] Log[
            1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/\[Rho]])/(2 R22^2) + (
          R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
            1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
          2 R22^2 \[Rho]^2) + (\[Rho]^2 Arg[

            1 - (R22 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]21])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]21)])/R22^2 + (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          R22^2 \[Rho]^2) - (\[Rho] Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22 (1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (
          R^2 Sin[\[Theta] - \[Theta]21] Sin[
            2 (\[Theta] - \[Theta]21)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/
             R^2)) - (\[Rho]^2 Arg[
            1 - (R22 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]22])/\[Rho]] Sin[
            2 (\[Theta] - \[Theta]22)])/R22^2 - (
          R^4 Arg[1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          R22^2 \[Rho]^2) + (\[Rho] Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22 (1 + R22^2/\[Rho]^2 - (
             2 R22 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (
          R^2 Sin[\[Theta] - \[Theta]22] Sin[
            2 (\[Theta] - \[Theta]22)])/(
          2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
             2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/R22^2) ((
             I R22 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             R22 Sin[\[Theta] - \[Theta]21])/\[Rho]) Derivative[1][
            Arg][1 - (R22 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]21])/\[Rho]] +
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
          1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/R22^2) ((
             I R22 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             R22 Sin[\[Theta] - \[Theta]22])/\[Rho]) Derivative[1][
            Arg][
            1 - (R22 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
             I R22 Sin[\[Theta] - \[Theta]22])/\[Rho]] -
          1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
             R22^2 \[Rho]^2)) ((
             I R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
            1][Arg][
            1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
             I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), 0};

J24tot[\[Rho]_, \[Theta]_,
   t_] := \[Sigma] {-(1/(2 \[Pi]))
        B02 \[Omega] Cos[\[CurlyPhi] +
        t \[CapitalOmega]] ((-\[Theta]21 + \[Theta]22) \[Rho] + (-R22 \
+ R21^3/(3 \[Rho]^2) + (8 \[Rho])/3) (Sin[\[Theta] - \[Theta]21] -
           Sin[\[Theta] - \[Theta]22]) + (R21^4/(4 \[Rho]^3) + (
           3 \[Rho])/
           4 - \[Rho] Log[R22/\[Rho]]) (Sin[
            2 (\[Theta] - \[Theta]21)] -
           Sin[2 (\[Theta] - \[Theta]22)]) -
        8 \[Rho] (1/3 Sin[\[Theta] - \[Theta]21] -
           1/2 Arg[
             1 - Cos[\[Theta] - \[Theta]21] +
              I Sin[\[Theta] - \[Theta]21]] Sin[\[Theta] - \
\[Theta]21]^2 + 3/32 Sin[2 (\[Theta] - \[Theta]21)] -
           1/3 Sin[\[Theta] - \[Theta]22] +
           1/2 Arg[
             1 - Cos[\[Theta] - \[Theta]22] +
              I Sin[\[Theta] - \[Theta]22]] Sin[\[Theta] - \
\[Theta]22]^2 - 3/32 Sin[2 (\[Theta] - \[Theta]22)]) -
        R21^2 (-((\[Rho] Arg[
              1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
               I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Cos[
              2 (\[Theta] - \[Theta]21)])/
            R21^2) + (\[Rho] Arg[
             1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Cos[
             2 (\[Theta] - \[Theta]22)])/R21^2 -
           Sin[\[Theta] - \[Theta]21]/(2 R21) + (
           R21 Sin[\[Theta] - \[Theta]21])/(3 \[Rho]^2) + (
           R21^2 Sin[2 (\[Theta] - \[Theta]21)])/(
           4 \[Rho]^3) - (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
              2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2) Sin[
             2 (\[Theta] - \[Theta]21)])/(
           4 R21^2 (1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (\[Rho] \
Log[1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]21)])/(2 R21^2) +
           Sin[\[Theta] - \[Theta]22]/(2 R21) - (
           R21 Sin[\[Theta] - \[Theta]22])/(3 \[Rho]^2) - (
           R21^2 Sin[2 (\[Theta] - \[Theta]22)])/(
           4 \[Rho]^3) + (\[Rho]^2 (-((2 R21^2)/\[Rho]^3) + (
              2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2) Sin[
             2 (\[Theta] - \[Theta]22)])/(
           4 R21^2 (1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) + (\[Rho] \
Log[1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]22)])/(2 R21^2) +
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
              R21^2) ((R21 Cos[\[Theta] - \[Theta]21])/\[Rho]^2 - (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]^2) Derivative[
             1][Arg][
             1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] -
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
              R21^2) ((R21 Cos[\[Theta] - \[Theta]22])/\[Rho]^2 - (
              I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]^2) Derivative[
             1][Arg][
             1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]]) -
        R21^2 ((R^4 Arg[
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
             2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^3) - (
           R^4 Arg[
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
             2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^3) + (
           R^2 Sin[\[Theta] - \[Theta]21])/(2 R21 \[Rho]^2) - (
           R^4 ((2 R21^2 \[Rho])/R^4 - (
              2 R21 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
             2 (\[Theta] - \[Theta]21)])/(
           4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
           R^4 Log[
             1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
             2 (\[Theta] - \[Theta]21)])/(2 R21^2 \[Rho]^3) - (
           R^2 Sin[\[Theta] - \[Theta]22])/(2 R21 \[Rho]^2) + (
           R^4 ((2 R21^2 \[Rho])/R^4 - (
              2 R21 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
             2 (\[Theta] - \[Theta]22)])/(
           4 R21^2 \[Rho]^2 (1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
           R^4 Log[
             1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
             2 (\[Theta] - \[Theta]22)])/(2 R21^2 \[Rho]^3) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
              R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]21])/
               R^2) + (I R21 Sin[\[Theta] - \[Theta]21])/
              R^2) Derivative[1][Arg][
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
              R21^2 \[Rho]^2)) (-((R21 Cos[\[Theta] - \[Theta]22])/
               R^2) + (I R21 Sin[\[Theta] - \[Theta]22])/
              R^2) Derivative[1][Arg][
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
        R22^2 ((\[Rho] Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Cos[
             2 (\[Theta] - \[Theta]21)])/
           R22^2 - (\[Rho] Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Cos[
             2 (\[Theta] - \[Theta]22)])/R22^2 +
           Sin[\[Theta] - \[Theta]21]/(
           2 R22) + (\[Rho] Sin[2 (\[Theta] - \[Theta]21)])/(
           2 R22^2) - (\[Rho]^2 ((2 \[Rho])/R22^2 - (
              2 Cos[\[Theta] - \[Theta]21])/R22) Sin[
             2 (\[Theta] - \[Theta]21)])/(
           4 R22^2 (1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]21])/
              R22)) - (\[Rho] Log[
             1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22] Sin[
             2 (\[Theta] - \[Theta]21)])/(2 R22^2) -
           Sin[\[Theta] - \[Theta]22]/(
           2 R22) - (\[Rho] Sin[2 (\[Theta] - \[Theta]22)])/(
           2 R22^2) + (\[Rho]^2 ((2 \[Rho])/R22^2 - (
              2 Cos[\[Theta] - \[Theta]22])/R22) Sin[
             2 (\[Theta] - \[Theta]22)])/(
           4 R22^2 (1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]22])/
              R22)) + (\[Rho] Log[
             1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22] Sin[
             2 (\[Theta] - \[Theta]22)])/(2 R22^2) +
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
              R22^2) (-(Cos[\[Theta] - \[Theta]21]/R22) + (
              I Sin[\[Theta] - \[Theta]21])/R22) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
              R22^2) (-(Cos[\[Theta] - \[Theta]22]/R22) + (
              I Sin[\[Theta] - \[Theta]22])/R22) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
        R22^2 ((R^4 Arg[
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Cos[
             2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^3) - (
           R^4 Arg[
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Cos[
             2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^3) + (
           R^2 Sin[\[Theta] - \[Theta]21])/(2 R22 \[Rho]^2) - (

           R^4 ((2 R22^2 \[Rho])/R^4 - (
              2 R22 Cos[\[Theta] - \[Theta]21])/R^2) Sin[
             2 (\[Theta] - \[Theta]21)])/(
           4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) + (
           R^4 Log[
             1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2] Sin[
             2 (\[Theta] - \[Theta]21)])/(2 R22^2 \[Rho]^3) - (
           R^2 Sin[\[Theta] - \[Theta]22])/(2 R22 \[Rho]^2) + (
           R^4 ((2 R22^2 \[Rho])/R^4 - (
              2 R22 Cos[\[Theta] - \[Theta]22])/R^2) Sin[
             2 (\[Theta] - \[Theta]22)])/(
           4 R22^2 \[Rho]^2 (1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) - (
           R^4 Log[
             1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2] Sin[
             2 (\[Theta] - \[Theta]22)])/(2 R22^2 \[Rho]^3) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
              R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]21])/
               R^2) + (I R22 Sin[\[Theta] - \[Theta]21])/
              R^2) Derivative[1][Arg][
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
              R22^2 \[Rho]^2)) (-((R22 Cos[\[Theta] - \[Theta]22])/
               R^2) + (I R22 Sin[\[Theta] - \[Theta]22])/
              R^2) Derivative[1][Arg][
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])), -(1/(
       2 \[Pi] \[Rho]))
        B02 \[Omega] Cos[\[CurlyPhi] +
        t \[CapitalOmega]] ((-(R21^3/(3 \[Rho])) - R22 \[Rho] + (
           4 \[Rho]^2)/3) (Cos[\[Theta] - \[Theta]21] -
           Cos[\[Theta] - \[Theta]22]) + (2 Cos[
             2 (\[Theta] - \[Theta]21)] -
           2 Cos[2 (\[Theta] - \[Theta]22)]) (-(R21^4/(
            8 \[Rho]^2)) + \[Rho]^2/8 -
           1/2 \[Rho]^2 Log[R22/\[Rho]]) -
        4 \[Rho]^2 (1/3 Cos[\[Theta] - \[Theta]21] +
           3/16 Cos[2 (\[Theta] - \[Theta]21)] -
           1/3 Cos[\[Theta] - \[Theta]22] -
           3/16 Cos[2 (\[Theta] - \[Theta]22)] -
           Arg[1 - Cos[\[Theta] - \[Theta]21] +
              I Sin[\[Theta] - \[Theta]21]] Cos[\[Theta] - \
\[Theta]21] Sin[\[Theta] - \[Theta]21] +

           Arg[1 - Cos[\[Theta] - \[Theta]22] +
              I Sin[\[Theta] - \[Theta]22]] Cos[\[Theta] - \
\[Theta]22] Sin[\[Theta] - \[Theta]22] -
           1/2 Sin[\[Theta] - \[Theta]21]^2 (I Cos[\[Theta] - \
\[Theta]21] + Sin[\[Theta] - \[Theta]21]) Derivative[1][Arg][
             1 - Cos[\[Theta] - \[Theta]21] +
              I Sin[\[Theta] - \[Theta]21]] +
           1/2 Sin[\[Theta] - \[Theta]22]^2 (I Cos[\[Theta] - \
\[Theta]22] + Sin[\[Theta] - \[Theta]22]) Derivative[1][Arg][
             1 - Cos[\[Theta] - \[Theta]22] +
              I Sin[\[Theta] - \[Theta]22]]) -
        R21^2 (-((R21 Cos[\[Theta] - \[Theta]21])/(
            3 \[Rho])) - (\[Rho] Cos[\[Theta] - \[Theta]21])/(
           2 R21) - (R21^2 Cos[2 (\[Theta] - \[Theta]21)])/(
           4 \[Rho]^2) + (R21 Cos[\[Theta] - \[Theta]22])/(
           3 \[Rho]) + (\[Rho] Cos[\[Theta] - \[Theta]22])/(2 R21) + (
           R21^2 Cos[2 (\[Theta] - \[Theta]22)])/(
           4 \[Rho]^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
             1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho]])/(
           2 R21^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[

             1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho]])/(
           2 R21^2) + (\[Rho]^2 Arg[
             1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]21)])/
           R21^2 - (\[Rho] Sin[\[Theta] - \[Theta]21] Sin[
             2 (\[Theta] - \[Theta]21)])/(
           2 R21 (1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]21])/\[Rho])) - (\[Rho]^2 \
Arg[1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]] Sin[
             2 (\[Theta] - \[Theta]22)])/
           R21^2 + (\[Rho] Sin[\[Theta] - \[Theta]22] Sin[
             2 (\[Theta] - \[Theta]22)])/(
           2 R21 (1 + R21^2/\[Rho]^2 - (
              2 R21 Cos[\[Theta] - \[Theta]22])/\[Rho])) +
           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
              R21^2) ((I R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              R21 Sin[\[Theta] - \[Theta]21])/\[Rho]) Derivative[1][
             Arg][1 - (R21 Cos[\[Theta] - \[Theta]21])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]21])/\[Rho]] -

           1/2 (1 - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
              R21^2) ((I R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
              R21 Sin[\[Theta] - \[Theta]22])/\[Rho]) Derivative[1][
             Arg][1 - (R21 Cos[\[Theta] - \[Theta]22])/\[Rho] + (
              I R21 Sin[\[Theta] - \[Theta]22])/\[Rho]]) -
        R21^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R21 \[Rho])) + (
           R^2 Cos[\[Theta] - \[Theta]22])/(2 R21 \[Rho]) - (
           R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
             1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
           2 R21^2 \[Rho]^2) + (
           R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[
             1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
           2 R21^2 \[Rho]^2) + (
           R^4 Arg[
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
             2 (\[Theta] - \[Theta]21)])/(R21^2 \[Rho]^2) - (
           R^2 Sin[\[Theta] - \[Theta]21] Sin[
             2 (\[Theta] - \[Theta]21)])/(
           2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
           R^4 Arg[
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
             2 (\[Theta] - \[Theta]22)])/(R21^2 \[Rho]^2) + (
           R^2 Sin[\[Theta] - \[Theta]22] Sin[
             2 (\[Theta] - \[Theta]22)])/(
           2 R21 \[Rho] (1 + (R21^2 \[Rho]^2)/R^4 - (
              2 R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
              R21^2 \[Rho]^2)) ((
              I R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
             1][Arg][
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
              R21^2 \[Rho]^2)) ((
              I R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
             1][Arg][
             1 - (R21 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R21 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2]) +
        R22^2 ((\[Rho] Cos[\[Theta] - \[Theta]21])/(
           2 R22) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/(
           2 R22^2) - (\[Rho] Cos[\[Theta] - \[Theta]22])/(
           2 R22) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/(
           2 R22^2) - (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)] Log[
             1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]21])/R22])/(
           2 R22^2) + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)] Log[
             1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22])/(
           2 R22^2) - (\[Rho]^2 Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] Sin[
             2 (\[Theta] - \[Theta]21)])/
           R22^2 - (\[Rho]^3 Sin[\[Theta] - \[Theta]21] Sin[
             2 (\[Theta] - \[Theta]21)])/(
           2 R22^3 (1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]21])/
              R22)) + (\[Rho]^2 Arg[
             1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]22])/R22] Sin[
             2 (\[Theta] - \[Theta]22)])/
           R22^2 + (\[Rho]^3 Sin[\[Theta] - \[Theta]22] Sin[
             2 (\[Theta] - \[Theta]22)])/(
           2 R22^3 (1 + \[Rho]^2/R22^2 - (
              2 \[Rho] Cos[\[Theta] - \[Theta]22])/R22)) +
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]21)])/
              R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]21])/
              R22 + (\[Rho] Sin[\[Theta] - \[Theta]21])/
              R22) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]21])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]21])/R22] -
           1/2 (-1 + (\[Rho]^2 Cos[2 (\[Theta] - \[Theta]22)])/
              R22^2) ((I \[Rho] Cos[\[Theta] - \[Theta]22])/
              R22 + (\[Rho] Sin[\[Theta] - \[Theta]22])/
              R22) Derivative[1][Arg][
             1 - (\[Rho] Cos[\[Theta] - \[Theta]22])/R22 + (
              I \[Rho] Sin[\[Theta] - \[Theta]22])/R22]) +
        R22^2 (-((R^2 Cos[\[Theta] - \[Theta]21])/(2 R22 \[Rho])) + (
           R^2 Cos[\[Theta] - \[Theta]22])/(2 R22 \[Rho]) - (
           R^4 Cos[2 (\[Theta] - \[Theta]21)] Log[
             1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2])/(
           2 R22^2 \[Rho]^2) + (
           R^4 Cos[2 (\[Theta] - \[Theta]22)] Log[

             1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2])/(
           2 R22^2 \[Rho]^2) + (
           R^4 Arg[
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] Sin[
             2 (\[Theta] - \[Theta]21)])/(R22^2 \[Rho]^2) - (
           R^2 Sin[\[Theta] - \[Theta]21] Sin[
             2 (\[Theta] - \[Theta]21)])/(
           2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2)) - (
           R^4 Arg[
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2] Sin[
             2 (\[Theta] - \[Theta]22)])/(R22^2 \[Rho]^2) + (
           R^2 Sin[\[Theta] - \[Theta]22] Sin[
             2 (\[Theta] - \[Theta]22)])/(
           2 R22 \[Rho] (1 + (R22^2 \[Rho]^2)/R^4 - (
              2 R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2)) +
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]21)])/(
              R22^2 \[Rho]^2)) ((
              I R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2) Derivative[
             1][Arg][

             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]21])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]21])/R^2] -
           1/2 (1 - (R^4 Cos[2 (\[Theta] - \[Theta]22)])/(
              R22^2 \[Rho]^2)) ((
              I R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2) Derivative[
             1][Arg][
             1 - (R22 \[Rho] Cos[\[Theta] - \[Theta]22])/R^2 + (
              I R22 \[Rho] Sin[\[Theta] - \[Theta]22])/R^2])),
     0} +  + \[Sigma] Cross[{0, \[Rho] \[Omega], 0}, {0, 0,
      B02 Cos[\[CapitalOmega] t + \[CurlyPhi]]}];

'''

In [ ]:

# ============================================================
# UTILIDADES DE PARSEO MATHEMATICA -> PYTHON/SYMPY
# ============================================================

def preprocess_mathematica_text(text: str) -> str:
    replacements = [
        (r'\[Rho]', 'rho'),
        (r'\[Theta]11', 'theta11'),
        (r'\[Theta]12', 'theta12'),
        (r'\[Theta]21', 'theta21'),
        (r'\[Theta]22', 'theta22'),
        (r'\[Theta]', 'theta'),
        (r'\[Sigma]', 'sigma'),
        (r'\[Omega]', 'omega'),
        (r'\[CapitalOmega]', 'Omega'),
        (r'\[CurlyPhi]', 'varphi'),
        (r'\[Pi]', 'Pi'),
    ]
    for old, new in replacements:
        text = text.replace(old, new)

    text = ' '.join(text.replace('\\\n', '').split())
    text = re.sub(r'Derivative\[\s*1\s*\]\[\s*Arg\s*\]\[', 'Derivative[1][Arg][', text)
    text = text.replace('+ +', '+')

    text = re.sub(
        r'Cross\[\{0,\s*rho\s*omega,\s*0\},\s*\{0,\s*0,\s*B01\s*Cos\[Omega\s*t\]\}\]',
        '{rho*omega*B01*Cos[Omega*t], 0}',
        text,
    )
    text = re.sub(
        r'Cross\[\{0,\s*rho\s*omega,\s*0\},\s*\{0,\s*0,\s*B02\s*Cos\[Omega\s*t\s*\+\s*varphi\]\}\]',
        '{rho*omega*B02*Cos[Omega*t + varphi], 0}',
        text,
    )

    return text


def replace_derivative_products(text: str) -> str:
    target = 'Derivative[1][Arg]['
    i = 0
    out: list[str] = []

    while True:
        j = text.find(target, i)
        if j == -1:
            out.append(text[i:])
            break

        k = j + len(target)
        depth = 1
        while k < len(text) and depth > 0:
            if text[k] == '[':
                depth += 1
            elif text[k] == ']':
                depth -= 1
            k += 1
        z_expr = text[j + len(target):k - 1]

        p = j - 1
        while p >= 0 and text[p].isspace():
            p -= 1

        if p >= 0 and text[p] == ')':
            depth = 1
            q = p - 1
            while q >= 0 and depth > 0:
                if text[q] == ')':
                    depth += 1
                elif text[q] == '(':
                    depth -= 1
                q -= 1
            a_start = q + 1
        elif p >= 0 and text[p] == ']':
            depth = 1
            q = p - 1
            while q >= 0 and depth > 0:
                if text[q] == ']':
                    depth += 1
                elif text[q] == '[':
                    depth -= 1
                q -= 1
            a_start = q + 1
        else:
            q = p
            while q >= 0 and (text[q].isalnum() or text[q] in '._'):
                q -= 1
            a_start = q + 1

        a_expr = text[a_start:p + 1]
        out.append(text[i:a_start])
        out.append(f'DArg[{a_expr}, {z_expr}]')
        i = k

    return ''.join(out)


def extract_function_bodies(text: str) -> dict[str, str]:
    names = ['J11tot', 'J12tot', 'J13tot', 'J14tot', 'J21tot', 'J22tot', 'J23tot', 'J24tot']
    bodies: dict[str, str] = {}

    for i, name in enumerate(names):
        next_name = names[i + 1] if i + 1 < len(names) else None
        if next_name is None:
            pattern = rf'{name}\[rho_, theta_, t_\] := (.*?);\s*$'
        else:
            pattern = rf'{name}\[rho_, theta_, t_\] := (.*?); {next_name}'
        match = re.search(pattern, text)
        if match is None:
            raise ValueError(f'No pude extraer la definición de {name}')
        bodies[name] = match.group(1)

    return bodies


def sympy_expr_to_vector(expr: sp.Expr) -> tuple[sp.Expr, sp.Expr]:
    if isinstance(expr, Tuple):
        if len(expr) == 2:
            return tuple(expr)
        if len(expr) == 3:
            return tuple(expr[:2])
        raise ValueError(f'Se esperaba un vector de 2 o 3 componentes y llegó uno de {len(expr)}')

    if expr.is_Add:
        parts = [sympy_expr_to_vector(arg) for arg in expr.args]
        return tuple(sum(part[i] for part in parts) for i in range(2))

    if expr.is_Mul:
        scalar = sp.Integer(1)
        vector = None
        for arg in expr.args:
            if isinstance(arg, Tuple):
                vector = tuple(arg[:2]) if len(arg) >= 2 else tuple(arg)
            else:
                scalar *= arg
        if vector is None:
            raise ValueError('Encontré un producto sin vector asociado.')
        return tuple(scalar * comp for comp in vector)

    raise TypeError(f'No pude convertir a vector la expresión de tipo {type(expr)}')



@lru_cache(maxsize=1)
def build_numeric_currents() -> dict[str, callable]:
    text = preprocess_mathematica_text(MATHEMATICA_TEXT)
    text = replace_derivative_products(text)
    bodies = extract_function_bodies(text)

    var_names = [
        'rho', 'theta', 't', 'sigma', 'Omega', 'omega', 'varphi',
        'B01', 'B02', 'R11', 'R12', 'R21', 'R22',
        'theta11', 'theta12', 'theta21', 'theta22', 'R'
    ]
    symbols = sp.symbols(' '.join(var_names))

    custom_modules = [
        {
            'Arg': np.angle,
            'DArg': lambda zp, z: np.imag(zp / z),
        },
        'numpy'
    ]

    functions: dict[str, callable] = {}
    for name, body in bodies.items():
        parsed = parse_mathematica(body)
        vec = sympy_expr_to_vector(parsed)
        functions[name] = lambdify(symbols, vec, modules=custom_modules)

    return functions


# ============================================================
# REGION GEOMETRY
# ============================================================

def wrap_angle(a: np.ndarray | float) -> np.ndarray | float:
    return np.mod(a, 2 * np.pi)


def angle_in_sector(theta: np.ndarray, theta_a: float, theta_b: float) -> np.ndarray:
    th = wrap_angle(theta)
    a = wrap_angle(theta_a)
    b = wrap_angle(theta_b)
    if a <= b:
        return (th >= a) & (th <= b)
    return (th >= a) | (th <= b)


def make_masks(rho: np.ndarray, theta: np.ndarray, p: dict) -> dict[str, np.ndarray]:
    eps = p.get('boundary_epsilon', max(1e-12, 1e-9 * p['R']))
    in_disk = rho < p['R'] - eps
    sector1 = angle_in_sector(theta, p['theta11'], p['theta12'])
    sector2 = angle_in_sector(theta, p['theta21'], p['theta22'])

    return {
        'J11tot': in_disk & (rho < p['R11'] - eps),
        'J12tot': in_disk & (rho > p['R11'] + eps) & (rho < p['R12'] - eps) & (~sector1),
        'J13tot': in_disk & (rho > p['R12'] + eps),
        'J14tot': in_disk & (rho > p['R11'] + eps) & (rho < p['R12'] - eps) & sector1,
        'J21tot': in_disk & (rho < p['R21'] - eps),
        'J22tot': in_disk & (rho > p['R21'] + eps) & (rho < p['R22'] - eps) & (~sector2),
        'J23tot': in_disk & (rho > p['R22'] + eps),
        'J24tot': in_disk & (rho > p['R21'] + eps) & (rho < p['R22'] - eps) & sector2,
    }


# ============================================================
# EVALUATION AND BASIS TRANSFORMATION
# ============================================================

def evaluate_piecewise_currents(x: np.ndarray, y: np.ndarray, t: float, p: dict) -> tuple[np.ndarray, np.ndarray]:
    funcs = build_numeric_currents()
    rho = np.hypot(x, y)
    theta = np.arctan2(y, x)
    rho_eval = np.where(rho < RHO_EPS, RHO_EPS, rho)

    masks = make_masks(rho, theta, p)
    Jr_total = np.zeros_like(rho, dtype=float)
    Jt_total = np.zeros_like(rho, dtype=float)

    ordered_args = [
        'rho', 'theta', 't', 'sigma', 'Omega', 'omega', 'varphi',
        'B01', 'B02', 'R11', 'R12', 'R21', 'R22',
        'theta11', 'theta12', 'theta21', 'theta22', 'R'
    ]

    for name, mask in masks.items():
        if not np.any(mask):
            continue

        f = funcs[name]
        values = {**p, 'rho': rho_eval[mask], 'theta': theta[mask], 't': t}
        args = [values[k] for k in ordered_args]
        with np.errstate(all='ignore'):
            jr_piece, jt_piece = f(*args)
        jr_piece = np.nan_to_num(np.real(jr_piece), nan=0.0, posinf=0.0, neginf=0.0)
        jt_piece = np.nan_to_num(np.real(jt_piece), nan=0.0, posinf=0.0, neginf=0.0)

        Jr_total[mask] += jr_piece
        Jt_total[mask] += jt_piece

    return Jr_total, Jt_total


def polar_to_cartesian_components(Jr: np.ndarray, Jt: np.ndarray, theta: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    Jx = Jr * np.cos(theta) - Jt * np.sin(theta)
    Jy = Jr * np.sin(theta) + Jt * np.cos(theta)
    return Jx, Jy


def build_grid(R: float, n: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    axis = np.linspace(-R, R, n)
    x, y = np.meshgrid(axis, axis)
    rho = np.hypot(x, y)
    theta = np.arctan2(y, x)
    inside = rho <= R
    return x, y, rho, theta, inside


# ============================================================
# PLOTS
# ============================================================

def plot_snapshots() -> Path:
    p = PARAMS.copy()

    x, y, rho, theta, inside = build_grid(p['R'], N_GRID)
    T = 2 * np.pi / p['Omega']
    times = np.linspace(0.0, T, N_SNAPSHOTS, endpoint=False)

    ncols = 3
    nrows = int(np.ceil(N_SNAPSHOTS / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=FIGSIZE, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    global_vmax = None
    mags = []
    fields = []
    for t in times:
        Jr, Jt = evaluate_piecewise_currents(x, y, t, p)
        Jx, Jy = polar_to_cartesian_components(Jr, Jt, theta)
        mag = np.sqrt(Jx**2 + Jy**2)
        mag = np.where(inside, mag, np.nan)
        mags.append(mag)
        fields.append((Jx, Jy))
        local = np.nanpercentile(mag, 98)
        global_vmax = local if global_vmax is None else max(global_vmax, local)

    for ax, t, mag, (Jx, Jy) in zip(axes, times, mags, fields):
        im = ax.pcolormesh(x, y, mag, shading='auto', cmap=CMAP, vmin=0, vmax=global_vmax)

        U = np.divide(Jx, mag, out=np.zeros_like(Jx), where=np.isfinite(mag) & (mag > 1e-14))
        V = np.divide(Jy, mag, out=np.zeros_like(Jy), where=np.isfinite(mag) & (mag > 1e-14))
        U = np.where(inside, U, np.nan)
        V = np.where(inside, V, np.nan)

        ax.quiver(
            x[::QUIVER_STRIDE, ::QUIVER_STRIDE],
            y[::QUIVER_STRIDE, ::QUIVER_STRIDE],
            U[::QUIVER_STRIDE, ::QUIVER_STRIDE],
            V[::QUIVER_STRIDE, ::QUIVER_STRIDE],
            pivot='mid',
            angles='xy',
            scale_units='xy',
            scale=18,
            width=0.0022,
            color=QUIVER_COLOR,
        )

        circle = plt.Circle((0, 0), p['R'], color='white', fill=False, lw=1.2)
        ax.add_patch(circle)
        ax.set_aspect('equal')
        ax.set_xlim(-p['R'], p['R'])
        ax.set_ylim(-p['R'], p['R'])
        ax.set_title(f't = {t:.4f}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    for ax in axes[len(times):]:
        ax.axis('off')

    fig.colorbar(im, ax=axes[:len(times)], shrink=0.9, label='|J|')
    fig.suptitle(f'Corrientes en un período: T = 2π/Ω = {T:.4f}', fontsize=14)
    fig.savefig(OUTPUT_FIG, dpi=180, bbox_inches='tight')
    plt.show()
    return OUTPUT_FIG


def make_gif() -> Path:
    from matplotlib.animation import FuncAnimation, PillowWriter

    p = PARAMS.copy()

    x, y, rho, theta, inside = build_grid(p['R'], N_GRID)
    T = 2 * np.pi / p['Omega']
    times = np.linspace(0.0, T, N_ANIMATION_FRAMES, endpoint=False)

    vmax = 0.0
    for t in times:
        Jr, Jt = evaluate_piecewise_currents(x, y, t, p)
        Jx, Jy = polar_to_cartesian_components(Jr, Jt, theta)
        mag = np.sqrt(Jx**2 + Jy**2)
        mag = np.where(inside, mag, np.nan)
        vmax = max(vmax, float(np.nanpercentile(mag, 98)))

    fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)

    Jr0, Jt0 = evaluate_piecewise_currents(x, y, times[0], p)
    Jx0, Jy0 = polar_to_cartesian_components(Jr0, Jt0, theta)
    mag0 = np.sqrt(Jx0**2 + Jy0**2)
    mag0 = np.where(inside, mag0, np.nan)

    U0 = np.divide(Jx0, mag0, out=np.zeros_like(Jx0), where=np.isfinite(mag0) & (mag0 > 1e-14))
    V0 = np.divide(Jy0, mag0, out=np.zeros_like(Jy0), where=np.isfinite(mag0) & (mag0 > 1e-14))
    U0 = np.where(inside, U0, np.nan)
    V0 = np.where(inside, V0, np.nan)

    img = ax.imshow(
        mag0,
        extent=[-p['R'], p['R'], -p['R'], p['R']],
        origin='lower',
        cmap=CMAP,
        vmin=0,
        vmax=vmax,
        interpolation='nearest',
    )
    q = ax.quiver(
        x[::QUIVER_STRIDE, ::QUIVER_STRIDE],
        y[::QUIVER_STRIDE, ::QUIVER_STRIDE],
        U0[::QUIVER_STRIDE, ::QUIVER_STRIDE],
        V0[::QUIVER_STRIDE, ::QUIVER_STRIDE],
        pivot='mid',
        angles='xy',
        scale_units='xy',
        scale=18,
        width=0.0022,
        color=QUIVER_COLOR,
    )
    fig.colorbar(img, ax=ax, label='|J|')

    circle = plt.Circle((0, 0), p['R'], color='white', fill=False, lw=1.2)
    ax.add_patch(circle)
    ax.set_aspect('equal')
    ax.set_xlim(-p['R'], p['R'])
    ax.set_ylim(-p['R'], p['R'])
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    def update(frame: int):
        t = times[frame]
        Jr, Jt = evaluate_piecewise_currents(x, y, t, p)
        Jx, Jy = polar_to_cartesian_components(Jr, Jt, theta)
        mag = np.sqrt(Jx**2 + Jy**2)
        mag = np.where(inside, mag, np.nan)

        U = np.divide(Jx, mag, out=np.zeros_like(Jx), where=np.isfinite(mag) & (mag > 1e-14))
        V = np.divide(Jy, mag, out=np.zeros_like(Jy), where=np.isfinite(mag) & (mag > 1e-14))
        U = np.where(inside, U, np.nan)
        V = np.where(inside, V, np.nan)

        img.set_data(mag)
        q.set_UVC(U[::QUIVER_STRIDE, ::QUIVER_STRIDE], V[::QUIVER_STRIDE, ::QUIVER_STRIDE])
        ax.set_title(f't = {t:.4f}   |   T = {T:.4f}')
        return img, q

    ani = FuncAnimation(fig, update, frames=len(times), interval=1000 / FPS, blit=False)
    ani.save(OUTPUT_GIF, writer=PillowWriter(fps=FPS))
    plt.close(fig)
    return OUTPUT_GIF


In [ ]:

print('Construyendo funciones numéricas desde el texto embebido de Mathematica...')
build_numeric_currents()
print('Generando snapshots...')
fig_path = plot_snapshots()
print(f'PNG guardado en: {fig_path.resolve()}')
print('Generando GIF...')
gif_path = make_gif()
print(f'GIF guardado en: {gif_path.resolve()}')
print('\nVista previa del GIF:')
display(IPyImage(filename=str(gif_path)))
